# DSPy Optimization for Farmer.CHAT Stitching Prompt

This notebook uses DSPy to optimize the Stitching prompt that converts structured agricultural facts into natural, conversational responses for farmers in Bihar, India.

## Overview
- **Goal**: Optimize the stitching prompt to maximize conversationality scores
- **Model**: Gemma (via Google AI Studio, Hugging Face, or Ollama)
- **Metric**: 6-dimension evaluation (Content Quality, Communication Style, Practical Advice, Safety & Credibility, Conversation Flow, Response Format)
- **Method**: DSPy optimizers (BootstrapFewShot, MIPRO) to find optimal prompt configurations

## 1. Setup and Installation

In [ ]:
!pip install -q dspy-ai together openai openpyxl

In [ ]:
import dspy
import json
import re
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import random
import os
import pandas as pd

# Initialize API keys
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

# Model configuration
GPT_MODEL = "gpt-5.1"  # For evaluation
GEMMA_MODEL = "google/gemma-3n-E4B-it"  # For generation
print(f"DSPy version: {dspy.__version__}")
print(f"Evaluation Model: {GPT_MODEL}")
print(f"Fact Stitching Model: {GEMMA_MODEL}")

DSPy version: 3.1.0
Evaluation Model: gpt-5.1
Fact Stitching Model: google/gemma-3n-E4B-it


## 2. Choose Your Gen and Eval Models

In [ ]:

# Configure Gemma for generation (via Together AI)
gemma_lm = dspy.LM(
    model=f"together_ai/{GEMMA_MODEL}",
    api_key=TOGETHER_API_KEY,
    max_tokens=10000,
    temperature=0.7  # Some creativity for generation
)

# Configure GPT for evaluation (via OpenAI)
gpt_eval = dspy.LM(
    model=f"openai/{GPT_MODEL}",
    api_key=OPENAI_API_KEY,
    max_tokens=10000,
    temperature=0.0  # Deterministic for evaluation
)

# Set Gemma as the default for generation
dspy.configure(lm=gemma_lm)

print("✓ Gemma configured for generation via Together AI")
print("✓ GPT-4o configured for evaluation via OpenAI")

✓ Gemma configured for generation via Together AI
✓ GPT-4o configured for evaluation via OpenAI


## 3. Test the Model Connection

In [ ]:
# # Quick test to ensure the model is working
# test_prompt = "What is the best time to plant rice in Bihar, India? Answer in one sentence."

# try:
#     response = gemma_lm(test_prompt)
#     print("✓ Model connection successful!")
#     print(f"\nTest response: {response}")
# except Exception as e:
#     print(f"✗ Model connection failed: {e}")

# Quick test to ensure the model is working
test_prompt = "What is the best time to plant rice in Bihar, India? Answer in one sentence."

try:
    response = gemma_lm(test_prompt)
    print("✓ Model connection successful!")
    print(f"\nTest response: {response}")
except Exception as e:
    print(f"✗ Model connection failed: {e}")

✓ Model connection successful!

Test response: ['The best time to plant rice in Bihar, India is generally between June and September, coinciding with the monsoon season. \n']


In [ ]:
# Load the RLHF SFT data with real validated examples
from google.colab import files

# Upload the Excel file
# print("Please upload 'RLHF SFT DATA with Facts Information.xlsx'")
uploaded = files.upload()
# Get the filename
excel_filename = list(uploaded.keys())[0]
print(f"✓ Uploaded: {excel_filename}")

# Load the Excel file
df = pd.read_excel(excel_filename, sheet_name='data.csv')
print(f"✓ Loaded {len(df)} total rows")

# Display columns
print(f"\nColumns: {df.columns.tolist()}")

Saving RLHF SFT DATA with Facts Information.xlsx to RLHF SFT DATA with Facts Information (1).xlsx
✓ Uploaded: RLHF SFT DATA with Facts Information (1).xlsx
✓ Loaded 8480 total rows

Columns: ['Unnamed: 0', 'crop_name', 'question', 'response', 'question_answer_id', 'user_id', 'incorrect_sections', 'rating', 'tags', 'additional_info', 'likes', 'dislikes', 'skipped', 'validation_status', 'id', 'created_at', 'updated_at', 'is_validated', 'payment_id', 'missing_info', 'gender_validated', 'using_separate_missing_info', 'corrected_response_ir_inc_cor_respo_plain_txt', 'Golden answers', 'gt_facts_list']


In [ ]:

# Filter for rows with valid gt_facts_list and Golden answers
df_valid = df[
    df['gt_facts_list'].notna() &
    df['Golden answers'].notna() &
    df['question'].notna()
].copy()

print(f"✓ Found {len(df_valid)} rows with complete data")

# ============================================================
# SAMPLE STRATEGY: Quality over Quantity
# ============================================================
# DSPy doesn't need 10,000 examples - it needs ~100 good ones
# to find the best 5-10 demonstrations for the prompt

SAMPLE_SIZE = 100  # Plenty for prompt optimization

random.seed(42)

# Optional: Stratified sampling by crop if you want diversity
if 'crop_name' in df_valid.columns and df_valid['crop_name'].notna().sum() > 0:
    # Try to get diverse crops
    crops = df_valid['crop_name'].dropna().unique()
    samples_per_crop = max(5, SAMPLE_SIZE // len(crops))

    sampled_dfs = []
    for crop in crops[:15]:  # Limit to 15 crop types
        crop_df = df_valid[df_valid['crop_name'] == crop]
        n_sample = min(samples_per_crop, len(crop_df))
        if n_sample > 0:
            sampled_dfs.append(crop_df.sample(n=n_sample, random_state=42))

    df_sample = pd.concat(sampled_dfs).drop_duplicates()

    # If we don't have enough, add random samples
    if len(df_sample) < SAMPLE_SIZE:
        remaining = df_valid[~df_valid.index.isin(df_sample.index)]
        extra = remaining.sample(n=min(SAMPLE_SIZE - len(df_sample), len(remaining)), random_state=42)
        df_sample = pd.concat([df_sample, extra])

    print(f"✓ Stratified sample: {len(df_sample)} examples across {df_sample['crop_name'].nunique()} crops")
else:
    # Random sampling
    df_sample = df_valid.sample(n=min(SAMPLE_SIZE, len(df_valid)), random_state=42)
    print(f"✓ Random sample: {len(df_sample)} examples")

# Preview
print(f"\n--- Sample Distribution ---")
if 'crop_name' in df_sample.columns:
    print(df_sample['crop_name'].value_counts().head(10))

✓ Found 8480 rows with complete data
✓ Stratified sample: 100 examples across 13 crops

--- Sample Distribution ---
crop_name
Paddy        10
Onion         9
Wheat         9
Brinjal       8
Lentil        8
Sugarcane     7
Maize         7
Tomato        7
Jute          7
Chilli        7
Name: count, dtype: int64


## 4. Define Data Structures

In [ ]:
@dataclass
class AgriculturalFact:
    """Represents a structured agricultural fact with metadata"""
    content: str
    category: str
    confidence: float
    bihar_relevance: str  # "high", "medium", "low"

    def to_dict(self) -> dict:
        # Convert bihar_relevance to numeric for consistency
        relevance_map = {"high": 0.9, "medium": 0.7, "low": 0.5}
        return {
            "content": self.content,
            "category": self.category,
            "confidence": self.confidence,
            "bihar_relevance": relevance_map.get(self.bihar_relevance, 0.8)
        }


@dataclass
class FarmerQuery:
    """Represents a farmer's question with context"""
    question: str
    crop: Optional[str] = None
    season: Optional[str] = None
    district: Optional[str] = None
    facts: List[AgriculturalFact] = None

    def facts_to_json(self) -> str:
        if not self.facts:
            return "[]"
        return json.dumps([f.to_dict() for f in self.facts], indent=2)

In [ ]:

def parse_facts_from_json(facts_str: str) -> List[Dict]:
    """Parse the gt_facts_list JSON string into a list of fact dicts"""
    try:
        if pd.isna(facts_str) or not facts_str:
            return []
        facts = json.loads(facts_str)
        # Normalize the facts structure
        normalized = []
        for f in facts:
            normalized.append({
                "content": f.get("fact", ""),
                "category": f.get("category", "general"),
                "confidence": f.get("confidence", 0.8),
                "bihar_relevance": 0.9 if f.get("bihar_relevance") == "high" else 0.7
            })
        return normalized
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        return []

def create_dspy_examples_from_df(df: pd.DataFrame) -> List[dspy.Example]:
    """
    Convert DataFrame rows to DSPy Examples with golden answers as labels.
    Only returns examples that have valid golden answers.
    """
    examples = []

    for idx, row in df.iterrows():
        try:
            # Parse facts
            facts = parse_facts_from_json(row['gt_facts_list'])
            if not facts:
                continue

            # Get question and golden answer
            question = str(row['question']).strip()
            golden_answer = str(row['Golden answers']).strip() if pd.notna(row['Golden answers']) else ""

            # Skip if no golden answer
            if not golden_answer:
                continue

            # Extract crop context
            crop_context = "Bihar, India"
            if pd.notna(row.get('crop_name')):
                crop_context = f"Crop: {row['crop_name']}, Bihar, India"

            # Create example WITH golden answer as the response label
            example = dspy.Example(
                farmer_question=question,
                agricultural_facts=json.dumps(facts, indent=2),
                crop_context=crop_context,
                response=golden_answer  # Human-validated answer for training
            ).with_inputs("farmer_question", "agricultural_facts", "crop_context")

            examples.append(example)

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            continue

    return examples

# Create all examples with golden answers
all_examples = create_dspy_examples_from_df(df_sample)
print(f"✓ Created {len(all_examples)} examples with golden answers")

# Proper train/val split - validation examples are HELD OUT
random.seed(42)
random.shuffle(all_examples)

split_idx = int(len(all_examples)-20)
train_set = all_examples[:split_idx]
val_set = all_examples[split_idx:]

print(f"\n  Training set: {len(train_set)} examples (golden answers used as few-shot demos)")
print(f"  Validation set: {len(val_set)} examples (model generates NEW responses, golden answers not shown)")
print(f"\n⚠️  Key: Model learns from train golden answers, evaluated on generating new responses for val set")

JSON parse error: Expecting value: line 1 column 1 (char 0)
JSON parse error: Expecting value: line 1 column 1 (char 0)
JSON parse error: Expecting value: line 1 column 1 (char 0)
✓ Created 97 examples with golden answers

  Training set: 77 examples (golden answers used as few-shot demos)
  Validation set: 20 examples (model generates NEW responses, golden answers not shown)

⚠️  Key: Model learns from train golden answers, evaluated on generating new responses for val set


In [ ]:
# ============================================================
# ANALYZE LANGUAGE DISTRIBUTION IN TRAINING DATA
# ============================================================

def detect_language(text):
    """Simple language detection based on character ranges"""
    if not text:
        return "unknown"

    # Count Devanagari characters (Hindi)
    devanagari_count = sum(1 for c in text if '\u0900' <= c <= '\u097F')
    total_alpha = sum(1 for c in text if c.isalpha())

    if total_alpha == 0:
        return "unknown"

    hindi_ratio = devanagari_count / total_alpha

    if hindi_ratio > 0.5:
        return "hindi"
    elif hindi_ratio > 0.1:
        return "hinglish"
    else:
        return "english"

# Analyze training set language distribution
lang_counts = {"english": 0, "hindi": 0, "hinglish": 0, "unknown": 0}
for ex in train_set:
    lang = detect_language(ex.farmer_question)
    lang_counts[lang] += 1

print("\n" + "="*60)
print("TRAINING DATA LANGUAGE DISTRIBUTION")
print("="*60)
for lang, count in lang_counts.items():
    pct = (count / len(train_set)) * 100
    print(f"  {lang.capitalize()}: {count} ({pct:.1f}%)")

# Check if language mixing in demos might be causing issues
print("\n⚠️  If Hindi/Hinglish examples dominate, the model may respond in Hindi")
print("   even for English queries. Consider filtering demos by language.")


TRAINING DATA LANGUAGE DISTRIBUTION
  English: 56 (72.7%)
  Hindi: 21 (27.3%)
  Hinglish: 0 (0.0%)
  Unknown: 0 (0.0%)

⚠️  If Hindi/Hinglish examples dominate, the model may respond in Hindi
   even for English queries. Consider filtering demos by language.


## 6. Define the Conversationality Evaluator

In [ ]:


class ConversationalityEvaluator:
    """Evaluates responses using the 6-dimension Farmer.CHAT guidelines"""

    EVAL_PROMPT_TEMPLATE = """You are an expert evaluator assessing agricultural advisory responses for farmers in Bihar, India.


**EVALUATION STANDARD: Farmer.CHAT Guidelines**


You are Farmer.CHAT, a knowledgeable agricultural advisor helping farmers in Bihar.


**YOUR ROLE:**
- Experienced agricultural extension worker with deep local knowledge
- Supportive mentor who understands smallholder farming challenges
- Culturally appropriate communication style for the region


**RESPONSE GUIDELINES:**


1. **Content Quality:**
 - Address the specific concern directly and practically
 - Provide actionable, region-appropriate advice
 - Include timing considerations based on current stage and season
 - Use local examples, varieties, and practices when relevant


2. **Communication Style:**
 - Warm, professional, and encouraging tone
 - Use simple, conversational language with appropriate cultural context
 - Explain technical concepts in simple terms
 - Avoid overly formal or academic language


3. **Practical Advice:**
 - Focus on low-cost, accessible solutions
 - Consider resource constraints of smallholder farmers
 - Mention local availability of inputs/resources
 - Include preventive measures when relevant


4. **Safety & Credibility:**
 - For chemical inputs: mention general categories rather than specific brands
 - Include safety precautions for handling chemicals/equipment
 - Encourage consultation with local experts for complex issues
 - Reference local agricultural departments or extension services


5. **Conversation Flow:**
 - Build on previous advice from chat history when relevant
 - Don't repeat information already covered
 - Ask clarifying questions if critical information is missing
 - Offer to elaborate on specific aspects if helpful


**RESPONSE FORMAT:**
Provide your complete response as a natural conversation. Structure your advice logically but don't force artificial formatting. Keep responses between 150-300 words.


**AVOID:**
- Generic advice that could apply to any crop/region
- Overly technical jargon without explanation
- Repetitive closing statements or tips
- Specific product recommendations without local context
- Assumptions about farmer's resources or experience level


---


**Question:** {question}


**Response to Evaluate:**
{response}


---


**Your Task:** Rate this response on 6 dimensions using a 1-5 scale based on how well it follows the Farmer.CHAT guidelines above.


**Evaluation Dimensions:**


1. **Content Quality** (1=poor, 5=excellent)
  - Does it address the specific concern directly and practically?
  - Is the advice actionable and region-appropriate?
  - Does it include timing considerations based on current stage and season?
  - Does it use local examples, varieties, and practices when relevant?


2. **Communication Style** (1=poor, 5=excellent)
  - Is the tone warm, professional, and encouraging?
  - Does it use simple, conversational language with appropriate cultural context?
  - Are technical concepts explained in simple terms?
  - Does it avoid overly formal or academic language?


3. **Practical Advice** (1=poor, 5=excellent)
  - Does it focus on low-cost, accessible solutions?
  - Does it consider resource constraints of smallholder farmers?
  - Does it mention local availability of inputs/resources?
  - Does it include preventive measures when relevant?


4. **Safety & Credibility** (1=poor, 5=excellent)
  - For chemical inputs: does it mention general categories rather than specific brands?
  - Does it include safety precautions for handling chemicals/equipment?
  - Does it encourage consultation with local experts for complex issues?
  - Does it reference local agricultural departments or extension services?


5. **Conversation Flow** (1=poor, 5=excellent)
  - Does it build on previous advice from chat history when relevant?
  - Does it avoid repeating information already covered?
  - Does it ask clarifying questions if critical information is missing?
  - Does it offer to elaborate on specific aspects if helpful?


6. **Response Format** (1=poor, 5=excellent)
  - Is it provided as a natural conversation?
  - Is the advice structured logically without artificial formatting?
  - Is the response length between 150-300 words?
  - Does it avoid the items listed in "AVOID" section?


---


**Output Format:** Provide your evaluation as a valid JSON object with this exact structure:


{{
   "content_quality": {{
       "score": <number 1-5>,
       "justification": "<2-3 sentence explanation with specific examples from the response>",
       "examples": ["<specific quote from response>", "<another quote if applicable>"]
   }},
   "communication_style": {{
       "score": <number 1-5>,
       "justification": "<2-3 sentence explanation with specific examples from the response>",
       "examples": ["<specific quote from response>"]
   }},
   "practical_advice": {{
       "score": <number 1-5>,
       "justification": "<2-3 sentence explanation with specific examples from the response>",
       "examples": ["<specific quote from response>"]
   }},
   "safety_credibility": {{
       "score": <number 1-5>,
       "justification": "<2-3 sentence explanation with specific examples from the response>",
       "examples": ["<specific quote from response>"]
   }},
   "conversation_flow": {{
       "score": <number 1-5>,
       "justification": "<2-3 sentence explanation with specific examples from the response>",
       "examples": ["<specific quote from response>"]
   }},
   "response_format": {{
       "score": <number 1-5>,
       "justification": "<2-3 sentence explanation with specific examples from the response>",
       "examples": ["<specific quote from response>"]
   }},
   "overall_score": <average of all 6 scores, rounded to 2 decimals>,
   "overall_assessment": "<3-4 sentence summary evaluating how well the response follows Farmer.CHAT guidelines>",
   "key_strengths": ["<strength 1 with reference to specific guideline>", "<strength 2 with reference to specific guideline>"],
   "areas_for_improvement": ["<improvement 1 with reference to specific guideline>", "<improvement 2 with reference to specific guideline>"]
}}


**Important:** Return ONLY the JSON object, no additional text before or after.
"""



    def __init__(self, eval_lm=None):
      self.eval_lm = eval_lm or gpt_eval

    def evaluate(self, question: str, response: str) -> Dict[str, Any]:
        """Evaluate a response and return scores"""

        prompt = self.EVAL_PROMPT_TEMPLATE.format(
            question=question,
            response=response
        )

        # Use the evaluation LM
        try:
            with dspy.context(lm=self.eval_lm):
                result = self.eval_lm(prompt)

            # Extract the response text
            if isinstance(result, list):
                result_text = result[0] if result else ""
            else:
                result_text = str(result)

            # Parse JSON from response
            json_match = re.search(r'\{[\s\S]*\}', result_text)
            if json_match:
                eval_result = json.loads(json_match.group())

                # Calculate overall if not present
                if 'overall_score' not in eval_result:
                    scores = []
                    for key in ['content_quality', 'communication_style', 'practical_advice',
                               'safety_credibility', 'conversation_flow', 'response_format']:
                        if key in eval_result and 'score' in eval_result[key]:
                            scores.append(eval_result[key]['score'])
                    if scores:
                        eval_result['overall_score'] = round(sum(scores) / len(scores), 1)

                return eval_result

        except json.JSONDecodeError as e:
            print(f"JSON parsing error: {e}")
        except Exception as e:
            print(f"Evaluation error: {e}")

        # Return default scores if parsing fails
        return {
            "content_quality": {"score": 3, "justification": "Parsing failed"},
            "communication_style": {"score": 3, "justification": "Parsing failed"},
            "practical_advice": {"score": 3, "justification": "Parsing failed"},
            "safety_credibility": {"score": 3, "justification": "Parsing failed"},
            "conversation_flow": {"score": 3, "justification": "Parsing failed"},
            "response_format": {"score": 3, "justification": "Parsing failed"},
            "overall_score": 3.0,
            "overall_assessment": "Evaluation parsing failed"
        }

    def get_overall_score(self, question: str, response: str) -> float:
        """Get just the overall score (for optimization)"""
        eval_result = self.evaluate(question, response)
        return eval_result.get("overall_score", 3.0)

# Initialize evaluator
evaluator = ConversationalityEvaluator()
print(f"✓ Evaluator initialized with {gpt_eval.model if hasattr(gpt_eval, 'model') else 'configured model'}")

✓ Evaluator initialized with openai/gpt-5.1


## 7. Define DSPy Signatures and Modules

In [ ]:
# # Define the signature for fact stitching
# class FactStitching(dspy.Signature):
#     """Synthesize structured agricultural facts into a natural, conversational response for a farmer in Bihar, India.

#     The response should:
#     - Address the farmer's question directly and practically
#     - Use warm, supportive language appropriate for smallholder farmers
#     - Integrate all relevant facts naturally without listing them mechanically
#     - Focus on actionable, low-cost solutions
#     - Include Bihar-specific context and local examples
#     - Be 150-300 words in length
#     - IMPORTANT: Respond in the SAME LANGUAGE as the farmer's question (English, Hindi, Hinglish etc.)
#     """

#     farmer_question: str = dspy.InputField(
#         desc="The farmer's question about their agricultural concern (may be in English, Hindi, or Hinglish)"
#     )
#     agricultural_facts: str = dspy.InputField(
#         desc="JSON array of structured facts with category, confidence, and Bihar relevance scores"
#     )
#     crop_context: str = dspy.InputField(
#         desc="Context about the crop, season, and location if available"
#     )

#     response: str = dspy.OutputField(
#         desc="A natural, conversational response in the SAME LANGUAGE as the farmer's question that synthesizes the facts into helpful farming advice"
#     )

# =============================================================================
# CELL: Full System Prompt as DSPy Signature
# =============================================================================

# Your original production system prompt
FULL_SYSTEM_PROMPT = """You are Farmer.CHAT, a knowledgeable agricultural advisor helping farmers.

**YOUR ROLE:**
- Experienced agricultural extension worker with deep local knowledge
- Supportive mentor who understands smallholder farming challenges
- Culturally appropriate communication style for the region making sure of user language as well

**RESPONSE GUIDELINES:**

1. **Content Quality:**
 - Address the specific concern directly and practically
 - Provide actionable, region-appropriate advice
 - Include timing considerations based on current stage and season
 - Use local examples, varieties, and practices when relevant

2. **Communication Style:**
 - Warm, professional, and encouraging tone
 - Use simple, conversational language with appropriate cultural context
 - Explain technical concepts in simple terms
 - Avoid overly formal or academic language

3. **Practical Advice:**
 - Focus on low-cost, accessible solutions
 - Consider resource constraints of smallholder farmers
 - Mention local availability of inputs/resources
 - Include preventive measures when relevant

4. **Safety & Credibility:**
 - For chemical inputs: mention general categories rather than specific brands
 - Include safety precautions for handling chemicals/equipment
 - Encourage consultation with local experts for complex issues
 - Reference local agricultural departments or extension services

5. **Conversation Flow:**
 - Build on previous advice from chat history when relevant
 - Don't repeat information already covered
 - Ask clarifying questions if critical information is missing
 - Offer to elaborate on specific aspects if helpful

**RESPONSE FORMAT:**
Provide your complete response as a natural conversation. Structure your advice logically but don't force artificial formatting. Keep responses between 150-300 words.

**AVOID:**
- Generic advice that could apply to any crop/region
- Overly technical jargon without explanation
- Repetitive closing statements or tips
- Specific product recommendations without local context
- Assumptions about farmer's resources or experience level

**ADDITIONAL RULES (from pattern analysis):**
- Reference local resources when relevant (KVK, agricultural extension centers, local mandi)
- End with encouragement or offer for follow-up
"""

# Create signature with FULL prompt
class FactStitching(dspy.Signature):
    __doc__ = FULL_SYSTEM_PROMPT

    farmer_question: str = dspy.InputField(
        desc="The farmer's question"
    )
    agricultural_facts: str = dspy.InputField(
        desc="JSON array of structured facts with category, confidence, and Bihar relevance scores"
    )
    crop_context: str = dspy.InputField(
        desc="Context about the crop, season, and location if available"
    )
    response: str = dspy.OutputField(
        desc="A natural, conversational response for the query."
    )

print("✓ FactStitching signature created with complete system prompt")
print(f"  Signature length: {len(FULL_SYSTEM_PROMPT)} chars")

✓ FactStitching signature created with complete system prompt
  Signature length: 2260 chars


In [ ]:
# class FarmerChatStitcher(dspy.Module):
#     """DSPy module for stitching agricultural facts into conversational responses"""

#     def __init__(self):
#         super().__init__()
#         self.generate_response = dspy.ChainOfThought(FactStitching)

#     def forward(self, farmer_question: str, agricultural_facts: str, crop_context: str = "") -> dspy.Prediction:
#         """Generate a stitched response from facts"""
#         result = self.generate_response(
#             farmer_question=farmer_question,
#             agricultural_facts=agricultural_facts,
#             crop_context=crop_context
#         )
#         return result

# # Create the base stitcher
# base_stitcher = FarmerChatStitcher()
# print("✓ FarmerChatStitcher module created")


class FarmerChatStitcher(dspy.Module):
    """DSPy module for stitching agricultural facts into conversational responses"""

    def __init__(self, use_cot=False):
        super().__init__()
        # LabeledFewShot works better with Predict
        # ChainOfThought adds reasoning but demos don't attach properly
        if use_cot:
            self.generate_response = dspy.ChainOfThought(FactStitching)
            print("Using ChainOfThought")
        else:
            self.generate_response = dspy.Predict(FactStitching)
            print("Using Predict")

    def forward(self, farmer_question: str, agricultural_facts: str, crop_context: str = "") -> dspy.Prediction:
        result = self.generate_response(
            farmer_question=farmer_question,
            agricultural_facts=agricultural_facts,
            crop_context=crop_context
        )
        return result

# For optimization, use Predict (demos attach properly)
base_stitcher = FarmerChatStitcher(use_cot=False)  # For baseline with CoT
optimization_stitcher = FarmerChatStitcher(use_cot=False)  # For LabeledFewShot

print("✓ FarmerChatStitcher module created")

Using Predict
Using Predict
✓ FarmerChatStitcher module created


### TODO: Interpretable COT -> Check prompts, Reduce prompt size to 15k tokens by summary

## 9. Define the Metric Function

In [ ]:
def conversationality_metric(example: dspy.Example, prediction: dspy.Prediction, trace=None) -> float:
    """DSPy metric function that evaluates response conversationality"""

    # Get the response from prediction
    response = getattr(prediction, 'response', '')
    if not response:
        return 0.0

    # Get the question from example
    question = example.farmer_question

    # Evaluate using our evaluator
    try:
        score = evaluator.get_overall_score(question, response)
        # Normalize to 0-1 range (scores are 1-5)
        normalized_score = (score - 1) / 4
        return normalized_score
    except Exception as e:
        print(f"Evaluation error: {e}")
        return 0.5  # Default middle score on error

print("✓ Metric function defined")

# def conversationality_metric(example: dspy.Example, prediction: dspy.Prediction, trace=None) -> bool:
#     """
#     DSPy metric that returns TRUE/FALSE for bootstrap optimization.
#     Bootstrap needs binary pass/fail, not continuous scores.
#     """

#     response = getattr(prediction, 'response', '')
#     if not response:
#         return False

#     question = example.farmer_question

#     try:
#         score = evaluator.get_overall_score(question, response)
#         # Return TRUE if score >= 4.5 (good quality threshold)
#         # Bootstrap will learn from examples that pass this threshold
#         is_good = score >= 4.5

#         # Optional: Print for debugging (comment out in production)
#         if trace:
#             print(f"  Score: {score:.2f} → {'PASS' if is_good else 'FAIL'}")

#         return is_good

#     except Exception as e:
#         print(f"Evaluation error: {e}")
#         return False

# print("✓ Binary metric function defined")

✓ Metric function defined


In [ ]:
# ============================================================================
# Enhanced metric with dimension weighting
# ============================================================================

def weighted_quality_metric(example: dspy.Example, prediction: dspy.Prediction, trace=None) -> bool:
    """
    More nuanced metric with weighted dimensions and higher threshold.
    Use this for stricter optimization.
    """

    response = getattr(prediction, 'response', '')
    if not response:
        return False

    question = example.farmer_question

    try:
        # Get detailed scores
        scores = evaluator.evaluate(question, response)

        # Weight dimensions by importance
        weighted_score = (
            scores['content_quality'] * 0.30 +        # Most important
            scores['communication_style'] * 0.25 +    # Very important
            scores['practical_advice'] * 0.25 +       # Very important
            scores.get('safety_credibility', 4) * 0.10 +
            scores.get('conversation_flow', 4) * 0.05 +
            scores.get('response_format', 4) * 0.05
        )

        # Higher threshold: 4.2/5 (84%) instead of 4.0/5 (80%)
        is_excellent = weighted_score >= 4.2

        if trace:
            print(f"  Weighted Score: {weighted_score:.2f} → {'PASS' if is_excellent else 'FAIL'}")

        return is_excellent

    except Exception as e:
        print(f"Evaluation error: {e}")
        return False

print("✓ Weighted quality metric defined")


✓ Weighted quality metric defined


## 10. Test Base Performance

In [ ]:
def evaluate_stitcher(stitcher: FarmerChatStitcher, examples: List[dspy.Example], name: str = "Stitcher") -> Dict:
    """Evaluate a stitcher on a set of examples"""
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")

    results = []
    detailed_scores = []

    for i, example in enumerate(examples):
        print(f"\n--- Example {i+1}: {example.farmer_question[:50]}...")

        try:
            # Generate response
            prediction = stitcher(
                farmer_question=example.farmer_question,
                agricultural_facts=example.agricultural_facts,
                crop_context=example.crop_context
            )

            response = prediction.response
            print(f"Response preview: {response[:150]}...")

            # Get detailed evaluation
            eval_result = evaluator.evaluate(example.farmer_question, response)

            overall_score = eval_result.get('overall_score', 3.0)
            results.append(overall_score)
            detailed_scores.append(eval_result)

            print(f"Overall Score: {overall_score:.2f}/5.0")
            print(f"  - Content Quality: {eval_result.get('content_quality', {}).get('score', 'N/A')}")
            print(f"  - Communication Style: {eval_result.get('communication_style', {}).get('score', 'N/A')}")
            print(f"  - Practical Advice: {eval_result.get('practical_advice', {}).get('score', 'N/A')}")

        except Exception as e:
            print(f"Error: {e}")
            results.append(3.0)

    avg_score = sum(results) / len(results) if results else 0
    print(f"\n{'='*60}")
    print(f"Average Score for {name}: {avg_score:.2f}/5.0")
    print(f"{'='*60}")

    return {
        'name': name,
        'average_score': avg_score,
        'individual_scores': results,
        'detailed_evaluations': detailed_scores
    }

# Test base performance on validation set
# print("\n" + "="*80)
# print("BASELINE EVALUATION")
# print("="*80)

# base_results = evaluate_stitcher(base_stitcher, val_set[3:10], "Base Stitcher (unoptimized)")

In [ ]:
# =============================================================================
# CELL 1: Define Patterns and Synthetic Demos
# =============================================================================

import dspy
import json

# Consolidated patterns from 100 golden answers
EXTRACTED_PATTERNS = [
    "Responses MUST match the query language (English→English, Hindi→Hindi, Hinglish→Hinglish)",
    "Use supportive and informative tone with actionable steps",
    "Integrate facts naturally within explanations - don't list them",
    "Encourage consulting local experts (KVK, कृषि विज्ञान केंद्र) and best practices",
    "Avoid overly technical jargon without explanation"
]

LANGUAGE_RULE = "CRITICAL: Response language MUST match query language exactly. English query = English response. Hindi query = Hindi response."

# Synthetic demos (compact, pattern-following, language-balanced)
SYNTHETIC_DEMOS = [
    {
        "farmer_question": "How can I manage pests in my rice field effectively?",
        "crop_context": "Crop: rice, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "Brown planthopper is a common pest in rice fields.", "category": "pest identification", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Neem oil is an effective natural pesticide for rice pests.", "category": "pest control", "confidence": 0.85, "bihar_relevance": 0.85},
            {"content": "Maintaining proper water levels can help reduce pest infestations.", "category": "cultural practice", "confidence": 0.88, "bihar_relevance": 0.9}
        ], indent=2),
        "response": "To manage pests like the brown planthopper in your rice field, consider using neem oil as a natural pesticide. Ensure your field has proper water levels to deter pests. Regular monitoring is crucial. For best results, consult local agricultural experts in Bihar for tailored advice. Implementing these steps can help protect your crop and increase yield. Remember, early intervention is key to effective pest management."
    },
    {
        "farmer_question": "गेहूं की सिंचाई के लिए सबसे अच्छा समय क्या है?",
        "crop_context": "Crop: wheat, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "गेहूं की पहली सिंचाई बुवाई के 20-25 दिन बाद करनी चाहिए।", "category": "irrigation timing", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "बिहार में गेहूं की सिंचाई के लिए 4-5 बार पानी देना आवश्यक होता है।", "category": "irrigation frequency", "confidence": 0.85, "bihar_relevance": 0.85},
            {"content": "सिंचाई के लिए 5-7 सेंटीमीटर पानी की गहराई पर्याप्त होती है।", "category": "water depth", "confidence": 0.88, "bihar_relevance": 0.9}
        ], indent=2, ensure_ascii=False),
        "response": "गेहूं की सिंचाई के लिए, पहली सिंचाई बुवाई के 20-25 दिन बाद करें। बिहार में 4-5 बार सिंचाई आवश्यक है। हर सिंचाई में 5-7 सेंटीमीटर पानी दें। इससे फसल की जड़ें मजबूत होंगी और उपज बढ़ेगी। स्थानीय कृषि विशेषज्ञों से सलाह लें और मौसम के अनुसार सिंचाई करें।"
    },
    {
        "farmer_question": "How can I improve the soil health for maize cultivation in Bihar?",
        "crop_context": "Crop: maize, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "Maize requires well-drained, fertile soil with a pH between 5.8 and 7.0.", "category": "soil requirements", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Green manures like dhaincha can improve soil fertility and structure.", "category": "soil improvement", "confidence": 0.85, "bihar_relevance": 0.9},
            {"content": "Regular soil testing helps in understanding nutrient needs and deficiencies.", "category": "soil testing", "confidence": 0.88, "bihar_relevance": 0.85}
        ], indent=2),
        "response": "Improving soil health for maize in Bihar involves several key practices. First, ensure the soil is well-drained and fertile, with a pH between 5.8 and 7.0. Incorporate green manures like dhaincha to enhance soil structure and fertility. Regular soil testing is crucial to identify nutrient deficiencies and adjust fertilization accordingly. Consider using organic compost to boost microbial activity and improve soil texture. Additionally, rotating crops with legumes can naturally replenish nitrogen levels. For best results, consult local agricultural experts who understand Bihar's specific soil conditions and climate."
    },
    {
        "farmer_question": "आलू की फसल में रोग प्रबंधन कैसे करें?",
        "crop_context": "Crop: potato, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "आलू की फसल में झुलसा रोग (Blight) एक सामान्य समस्या है, जो पत्तियों पर काले धब्बे बनाती है।", "category": "disease", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "झुलसा रोग को नियंत्रित करने के लिए मैन्कोजेब या क्लोरोथैलोनिल जैसे फफूंदनाशक का छिड़काव करें।", "category": "treatment", "confidence": 0.85, "bihar_relevance": 0.85},
            {"content": "बिहार की मिट्टी और जलवायु आलू की खेती के लिए उपयुक्त है, लेकिन रोग प्रबंधन के लिए उचित जल निकासी आवश्यक है।", "category": "soil and climate", "confidence": 0.88, "bihar_relevance": 0.9}
        ], indent=2, ensure_ascii=False),
        "response": "आलू की फसल में रोग प्रबंधन के लिए सबसे पहले झुलसा रोग की पहचान करें। यह रोग पत्तियों पर काले धब्बे बनाता है। इसे नियंत्रित करने के लिए मैन्कोजेब या क्लोरोथैलोनिल जैसे फफूंदनाशक का छिड़काव करें। छिड़काव से पहले स्थानीय कृषि विशेषज्ञ से सलाह लें। बिहार की जलवायु आलू की खेती के लिए अनुकूल है, लेकिन रोग से बचाव के लिए खेत में उचित जल निकासी सुनिश्चित करें। फसल चक्र अपनाएं और आलू की फसल के बाद दलहन उगाएं।"
    },
    {
        "farmer_question": "Organic farming ke liye vegetables kaise ugayein, koi tips hain?",
        "crop_context": "Crop: vegetables, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "Organic farming mein chemical fertilizers ki jagah compost aur green manure ka istemal hota hai.", "category": "organic practices", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Neem oil aur cow urine jaise natural pesticides insects ko control karne mein madadgar hote hain.", "category": "pest management", "confidence": 0.85, "bihar_relevance": 0.85},
            {"content": "Crop rotation aur intercropping se soil fertility aur pest control mein sudhar hota hai.", "category": "soil management", "confidence": 0.88, "bihar_relevance": 0.9}
        ], indent=2),
        "response": "Organic farming ke liye, sabse pehle compost aur green manure ka istemal karein. Neem oil aur cow urine jaise natural pesticides se insects ko control karein. Crop rotation aur intercropping se mitti ki fertility aur pest control mein sudhar hota hai. Apne ilake ke agricultural expert se salah lein aur local practices ko follow karein. Yeh steps aapko organic farming mein safalta dilayenge."
    },
    {
        "farmer_question": "How do I select the best seeds for growing pulses in Bihar?",
        "crop_context": "Crop: pulses, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "Pulses thrive in well-drained loamy soil with a pH of 6.0 to 7.5.", "category": "soil", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Certified seeds have higher germination rates and disease resistance.", "category": "seed quality", "confidence": 0.85, "bihar_relevance": 0.85},
            {"content": "Varieties like Pusa Vishal and Pusa Arhar 16 are suitable for Bihar's climate.", "category": "variety", "confidence": 0.88, "bihar_relevance": 0.9}
        ], indent=2),
        "response": "For selecting pulse seeds, choose certified seeds to ensure high germination and disease resistance. Varieties like Pusa Vishal and Pusa Arhar 16 are well-suited for Bihar's climate. Ensure your soil is loamy and well-drained, with a pH between 6.0 and 7.5. Consult local agricultural experts for personalized advice. Regular monitoring of soil conditions and seedling growth will help in achieving a successful yield."
    },
    {
        "farmer_question": "गन्ने की फसल को मौसम से कैसे सुरक्षित रखें?",
        "crop_context": "Crop: sugarcane, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "गन्ने की फसल को तेज हवाओं से बचाने के लिए खेत के चारों ओर वायु अवरोधक पौधे लगाना चाहिए।", "category": "crop protection", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "गन्ने की फसल को अधिक बारिश से बचाने के लिए उचित जल निकासी प्रणाली का होना आवश्यक है।", "category": "water management", "confidence": 0.85, "bihar_relevance": 0.85},
            {"content": "गन्ने की फसल के लिए 20-30 डिग्री सेल्सियस का तापमान सबसे उपयुक्त होता है।", "category": "climate", "confidence": 0.88, "bihar_relevance": 0.9}
        ], indent=2, ensure_ascii=False),
        "response": "गन्ने की फसल को मौसम के प्रभाव से बचाने के लिए कुछ महत्वपूर्ण कदम उठाए जा सकते हैं। सबसे पहले, तेज हवाओं से सुरक्षा के लिए खेत के चारों ओर वायु अवरोधक पौधे जैसे बबूल या नीम लगाएं। अधिक बारिश के दौरान जल निकासी प्रणाली का सही प्रबंधन करें ताकि पानी खेत में न रुके। गन्ने की फसल के लिए 20-30 डिग्री सेल्सियस का तापमान उपयुक्त होता है। स्थानीय कृषि विशेषज्ञों से परामर्श करें और उनके सुझावों का पालन करें।"
    },
    {
        "farmer_question": "Mustard ke liye kaunsa fertilizer best hai aur kitna istemal karna chahiye?",
        "crop_context": "Crop: mustard, Bihar, India",
        "agricultural_facts": json.dumps([
            {"content": "Mustard ki fasal ke liye nitrogen, phosphorus, aur potassium zaroori hote hain.", "category": "nutrient requirement", "confidence": 0.9, "bihar_relevance": 0.9},
            {"content": "Bihar ke mitti mein organic matter kam hota hai, isliye compost ya green manure ka istemal faydemand hai.", "category": "soil condition", "confidence": 0.85, "bihar_relevance": 0.9},
            {"content": "Mustard ke liye 60 kg N, 30 kg P2O5, aur 20 kg K2O per hectare recommended hai.", "category": "fertilizer recommendation", "confidence": 0.88, "bihar_relevance": 0.85}
        ], indent=2),
        "response": "Mustard ki fasal ke liye nitrogen, phosphorus, aur potassium ka sahi santulan zaroori hai. Aap 60 kg nitrogen, 30 kg phosphorus (P2O5), aur 20 kg potassium (K2O) per hectare istemal karein. Bihar ke mitti mein organic matter kam hota hai, isliye compost ya green manure ka istemal bhi karein. Behtar fasal ke liye local krishi vibhag se salah lein aur mitti ki jaanch karwa kar fertilizer ka istemal karein."
    }
]

print(f"✓ Loaded {len(SYNTHETIC_DEMOS)} synthetic demos")
print(f"  - English: {sum(1 for d in SYNTHETIC_DEMOS if 'How' in d['farmer_question'] or 'What' in d['farmer_question'])}")
print(f"  - Hindi: {sum(1 for d in SYNTHETIC_DEMOS if any(ord(c) >= 0x0900 and ord(c) <= 0x097F for c in d['farmer_question']))}")
print(f"  - Hinglish: {sum(1 for d in SYNTHETIC_DEMOS if 'ke liye' in d['farmer_question'].lower() or 'kaise' in d['farmer_question'].lower())}")

✓ Loaded 8 synthetic demos
  - English: 3
  - Hindi: 3
  - Hinglish: 2


In [ ]:
# =============================================================================
# CELL 2: PatternEnhancedFactStitching - Reuse FULL_SYSTEM_PROMPT
# =============================================================================

class PatternEnhancedFactStitching(dspy.Signature):
    __doc__ = FULL_SYSTEM_PROMPT  # Using the exact same prompt as FactStitching

    farmer_question: str = dspy.InputField(
        desc="The farmer's question"
    )
    agricultural_facts: str = dspy.InputField(
        desc="JSON array of structured facts with category, confidence, and Bihar relevance scores"
    )
    crop_context: str = dspy.InputField(
        desc="Context about the crop, season, and location if available"
    )
    response: str = dspy.OutputField(
        desc="A natural, conversational response in the SAME LANGUAGE as the query"
    )

print("✓ PatternEnhancedFactStitching defined (uses FULL_SYSTEM_PROMPT)")
print(f"  Signature length: {len(FULL_SYSTEM_PROMPT)} chars")

✓ PatternEnhancedFactStitching defined (uses FULL_SYSTEM_PROMPT)
  Signature length: 2260 chars


In [ ]:
# =============================================================================
# CELL 3: Create Pattern-Enhanced Module with Manual Demo Injection
# =============================================================================

class PatternEnhancedStitcher(dspy.Module):
    def __init__(self, demos=None):
        super().__init__()
        self.generate_response = dspy.ChainOfThought(PatternEnhancedFactStitching)

        if demos:
            self._inject_demos(demos)

    def _inject_demos(self, demos):
        demo_examples = [
            dspy.Example(
                farmer_question=d['farmer_question'],
                agricultural_facts=d['agricultural_facts'],
                crop_context=d['crop_context'],
                response=d['response']
            ).with_inputs('farmer_question', 'agricultural_facts', 'crop_context')
            for d in demos
        ]
        self.generate_response.demos = demo_examples
        print(f"  ✓ Injected {len(demo_examples)} demos")

    def forward(self, farmer_question, agricultural_facts, crop_context=""):
        return self.generate_response(
            farmer_question=farmer_question,
            agricultural_facts=agricultural_facts,
            crop_context=crop_context
        )


# Create the pattern-enhanced stitcher with synthetic demos
pattern_enhanced_stitcher = PatternEnhancedStitcher(demos=SYNTHETIC_DEMOS)
print("✓ PatternEnhancedStitcher created with synthetic demos")

# Verification
print("\n" + "="*80)
print("VERIFICATION:")

base_demos = len(getattr(base_stitcher.generate_response, 'demos', []))
enhanced_demos = len(getattr(pattern_enhanced_stitcher.generate_response, 'demos', []))

print(f"  Base demos:     {base_demos}")
print(f"  Enhanced demos: {enhanced_demos}")

if base_demos != enhanced_demos:
    print("  ✓ Different demo counts - Configuration correct!")
else:
    print("  ⚠️ Same demo count - Check configuration")

print("="*80)

  ✓ Injected 8 demos
✓ PatternEnhancedStitcher created with synthetic demos

VERIFICATION:
  Base demos:     0
  Enhanced demos: 8
  ✓ Different demo counts - Configuration correct!


In [ ]:
# =============================================================================
# CELL 5: Batch Evaluation - WITH AGGRESSIVE HISTORY CLEARING
# =============================================================================

import json
from datetime import datetime
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='pydantic')

print("\n" + "="*80)
print("BATCH EVALUATION: BASE vs PATTERN-ENHANCED (20 Examples)")
print("="*80)

test_set = all_examples[:20]
print(f"\nTesting on {len(test_set)} examples")

# Storage for results
results_data = {
    "timestamp": datetime.now().isoformat(),
    "num_examples": len(test_set),
    "base_stitcher": {
        "examples": [],
        "scores": [],
        "average_score": 0.0
    },
    "pattern_enhanced_stitcher": {
        "examples": [],
        "scores": [],
        "average_score": 0.0
    },
    "comparison": {}
}

# Evaluate base stitcher
print("\n" + "-"*80)
print("EVALUATING BASE STITCHER (Unoptimized)")
print("-"*80)

for i, example in enumerate(test_set):
    print(f"\n[{i+1}/{len(test_set)}] Processing: {example.farmer_question[:60]}...")

    # AGGRESSIVE HISTORY CLEARING
    if hasattr(base_stitcher.generate_response, 'history'):
        base_stitcher.generate_response.history = []

    # Clear LM-level history if it exists
    if hasattr(gemma_lm, 'history'):
        gemma_lm.history = []

    # Clear history from inner predictors
    try:
        for pred in base_stitcher.generate_response.predictors():
            if hasattr(pred, 'history'):
                pred.history = []
    except:
        pass

    try:
        # Generate response with fresh context
        with dspy.context(lm=gemma_lm):
            prediction = base_stitcher(
                farmer_question=example.farmer_question,
                agricultural_facts=example.agricultural_facts,
                crop_context=example.crop_context
            )

        response = prediction.response

        print(f"\nGenerated Response Preview:")
        print(f"  {response[:200]}...")

        # Evaluate
        eval_result = evaluator.evaluate(example.farmer_question, response)
        overall_score = eval_result.get('overall_score', 3.0)

        # Store results
        results_data["base_stitcher"]["examples"].append({
            "id": i,
            "question": example.farmer_question,
            "agricultural_facts": example.agricultural_facts,
            "crop_context": example.crop_context,
            "generated_response": response,
            "evaluation": eval_result,
            "overall_score": overall_score
        })
        results_data["base_stitcher"]["scores"].append(overall_score)

        print(f"  Score: {overall_score:.2f}/5.0")

    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback
        traceback.print_exc()
        results_data["base_stitcher"]["scores"].append(3.0)

# Calculate base average
base_avg = sum(results_data["base_stitcher"]["scores"]) / len(results_data["base_stitcher"]["scores"])
results_data["base_stitcher"]["average_score"] = round(base_avg, 2)

print("\n" + "="*80)
print(f"BASE STITCHER AVERAGE: {base_avg:.2f}/5.0")
print("="*80)

# Evaluate pattern-enhanced stitcher
print("\n" + "-"*80)
print("EVALUATING PATTERN-ENHANCED STITCHER")
print("-"*80)

for i, example in enumerate(test_set):
    print(f"\n[{i+1}/{len(test_set)}] Processing: {example.farmer_question[:60]}...")

    # AGGRESSIVE HISTORY CLEARING
    if hasattr(pattern_enhanced_stitcher.generate_response, 'history'):
        pattern_enhanced_stitcher.generate_response.history = []

    # Clear LM-level history
    if hasattr(gemma_lm, 'history'):
        gemma_lm.history = []

    # Clear history from inner predictors
    try:
        for pred in pattern_enhanced_stitcher.generate_response.predictors():
            if hasattr(pred, 'history'):
                pred.history = []
    except:
        pass

    try:
        # Generate response with fresh context
        with dspy.context(lm=gemma_lm):
            prediction = pattern_enhanced_stitcher(
                farmer_question=example.farmer_question,
                agricultural_facts=example.agricultural_facts,
                crop_context=example.crop_context
            )

        response = prediction.response

        print(f"\nGenerated Response Preview:")
        print(f"  {response[:200]}...")

        # Evaluate
        eval_result = evaluator.evaluate(example.farmer_question, response)
        overall_score = eval_result.get('overall_score', 3.0)

        # Store results
        results_data["pattern_enhanced_stitcher"]["examples"].append({
            "id": i,
            "question": example.farmer_question,
            "agricultural_facts": example.agricultural_facts,
            "crop_context": example.crop_context,
            "generated_response": response,
            "evaluation": eval_result,
            "overall_score": overall_score
        })
        results_data["pattern_enhanced_stitcher"]["scores"].append(overall_score)

        print(f"  Score: {overall_score:.2f}/5.0")

    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback
        traceback.print_exc()
        results_data["pattern_enhanced_stitcher"]["scores"].append(3.0)

# Calculate enhanced average
enhanced_avg = sum(results_data["pattern_enhanced_stitcher"]["scores"]) / len(results_data["pattern_enhanced_stitcher"]["scores"])
results_data["pattern_enhanced_stitcher"]["average_score"] = round(enhanced_avg, 2)

print("\n" + "="*80)
print(f"PATTERN-ENHANCED STITCHER AVERAGE: {enhanced_avg:.2f}/5.0")
print("="*80)

# Generate comparison report
improvement = enhanced_avg - base_avg
improvement_pct = (improvement / base_avg) * 100 if base_avg > 0 else 0

# Count wins/losses/ties
wins = sum(1 for i in range(len(test_set))
           if results_data["pattern_enhanced_stitcher"]["scores"][i] > results_data["base_stitcher"]["scores"][i])
losses = sum(1 for i in range(len(test_set))
             if results_data["pattern_enhanced_stitcher"]["scores"][i] < results_data["base_stitcher"]["scores"][i])
ties = sum(1 for i in range(len(test_set))
           if results_data["pattern_enhanced_stitcher"]["scores"][i] == results_data["base_stitcher"]["scores"][i])

# Store comparison
results_data["comparison"] = {
    "base_average": round(base_avg, 2),
    "enhanced_average": round(enhanced_avg, 2),
    "improvement": round(improvement, 2),
    "improvement_percentage": round(improvement_pct, 2),
    "wins": wins,
    "losses": losses,
    "ties": ties,
    "total_examples": len(test_set)
}

# Print comparison report
print("\n" + "="*80)
print("COMPARISON REPORT")
print("="*80)
print(f"Base Stitcher Average:           {base_avg:.2f}/5.0")
print(f"Pattern-Enhanced Average:        {enhanced_avg:.2f}/5.0")
print(f"Improvement:                     {improvement:+.2f} ({improvement_pct:+.1f}%)")
print(f"\nHead-to-Head:")
print(f"  Pattern-Enhanced Wins:         {wins}/{len(test_set)}")
print(f"  Pattern-Enhanced Losses:       {losses}/{len(test_set)}")
print(f"  Ties:                          {ties}/{len(test_set)}")
print("="*80)

# Save to JSON file
output_filename = f"evaluation_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(results_data, f, indent=2, ensure_ascii=False)

print(f"\n✓ Results saved to: {output_filename}")
print(f"  File size: {len(json.dumps(results_data))} bytes")


BATCH EVALUATION: BASE vs PATTERN-ENHANCED (20 Examples)

Testing on 20 examples

--------------------------------------------------------------------------------
EVALUATING BASE STITCHER (Unoptimized)
--------------------------------------------------------------------------------

[1/20] Processing: Are there any mustard varieties with unique features, such a...

Generated Response Preview:
  Namaste! Yes, there are definitely mustard varieties that offer unique features like enhanced flavor or oil content. In Bihar, you have some excellent options to choose from.

You can consider Pusa Mu...
  Score: 4.17/5.0

[2/20] Processing: Can lentils be grown on sandy loam soils?...

Generated Response Preview:
  Namaste! Yes, lentils *can* be grown on sandy loam soils in Bihar, but there are a few things to keep in mind. Sandy loam soils are good if they have good drainage, which is important for lentils to a...
  Score: 4.17/5.0

[3/20] Processing: क्या जैविक विधियों के कारण आलू उत्पादन मे

Traceback (most recent call last):
  File "/tmp/ipython-input-2614860134.py", line 141, in <cell line: 0>
    print(f"  {response[:200]}...")
               ~~~~~~~~^^^^^^
TypeError: 'NoneType' object is not subscriptable


In [ ]:
# =============================================================================
# Extract Prompts and Update JSON - FINAL FIX (Inspect DSPy Internals)
# =============================================================================

import json
from datetime import datetime
import inspect

# Load your existing results
with open('evaluation_results_20260112_085427.json', 'r', encoding='utf-8') as f:
    results_data = json.load(f)

print("\n" + "="*80)
print("EXTRACTING PROMPTS FROM MODELS")
print("="*80)

# =============================================================================
# Deep inspection function
# =============================================================================

def deep_inspect_predictor(predictor, model_name="Model"):
    """Deep inspection of DSPy predictor to find signature"""
    print(f"\n{'='*80}")
    print(f"DEEP INSPECTION: {model_name}")
    print(f"{'='*80}")

    result = {
        "predictor_type": type(predictor).__name__,
        "signature_name": "Unknown",
        "docstring": "",
        "input_fields": {},
        "output_fields": {},
        "uses_chain_of_thought": False,
        "few_shot_demos": [],
        "num_demos": 0
    }

    # Check predictor type
    result["uses_chain_of_thought"] = "ChainOfThought" in result["predictor_type"]

    # Method 1: Check predictors() method
    print("\n1. Checking .predictors()...")
    try:
        preds = list(predictor.predictors())
        print(f"   Found {len(preds)} sub-predictors")
        if preds:
            # ChainOfThought wraps a Predict - get the inner one
            inner_pred = preds[0]
            print(f"   Inner predictor type: {type(inner_pred).__name__}")

            if hasattr(inner_pred, 'signature'):
                sig = inner_pred.signature
                print(f"   ✓ Found signature in inner predictor!")
                result["signature_name"] = sig.__name__ if hasattr(sig, '__name__') else str(type(sig).__name__)
                result["docstring"] = getattr(sig, '__doc__', '') or ''

                # Extract fields
                if hasattr(sig, 'input_fields'):
                    for name, field in sig.input_fields.items():
                        desc = ""
                        try:
                            if hasattr(field, 'json_schema_extra') and field.json_schema_extra:
                                desc = field.json_schema_extra.get('desc', '')
                        except:
                            pass
                        result["input_fields"][name] = {"description": desc}

                if hasattr(sig, 'output_fields'):
                    for name, field in sig.output_fields.items():
                        desc = ""
                        try:
                            if hasattr(field, 'json_schema_extra') and field.json_schema_extra:
                                desc = field.json_schema_extra.get('desc', '')
                        except:
                            pass
                        result["output_fields"][name] = {"description": desc}
    except Exception as e:
        print(f"   Error: {e}")

    # Method 2: Check for demos
    print("\n2. Checking for few-shot demos...")
    if hasattr(predictor, 'demos') and predictor.demos:
        result["num_demos"] = len(predictor.demos)
        print(f"   ✓ Found {len(predictor.demos)} demos")

        for i, demo in enumerate(predictor.demos, 1):
            demo_dict = {
                "demo_id": i,
                "farmer_question": getattr(demo, 'farmer_question', 'N/A'),
                "agricultural_facts": getattr(demo, 'agricultural_facts', 'N/A')[:200] + "..."
                    if len(getattr(demo, 'agricultural_facts', '')) > 200
                    else getattr(demo, 'agricultural_facts', 'N/A'),
                "crop_context": getattr(demo, 'crop_context', 'N/A'),
                "response": getattr(demo, 'response', 'N/A')
            }
            result["few_shot_demos"].append(demo_dict)
    else:
        print(f"   No demos found")

    # Method 3: Inspect history for actual prompts used
    print("\n3. Checking prediction history...")
    try:
        if hasattr(predictor, 'history') and predictor.history:
            last_entry = predictor.history[-1]
            print(f"   Found {len(predictor.history)} history entries")
            print(f"   Last entry type: {type(last_entry)}")

            # Try to extract prompt from history
            if hasattr(last_entry, 'prompt'):
                result["last_prompt_used"] = str(last_entry.prompt)[:500]
                print(f"   ✓ Captured last prompt used (first 200 chars)")
    except Exception as e:
        print(f"   Error: {e}")

    return result

# =============================================================================
# Extract from Base Stitcher
# =============================================================================

print("\n" + "-"*80)
print("EXTRACTING: BASE STITCHER")
print("-"*80)

base_prompt = deep_inspect_predictor(base_stitcher.generate_response, "Base Stitcher")

# Print summary
print(f"\n{'='*80}")
print("BASE STITCHER SUMMARY")
print(f"{'='*80}")
print(f"Signature: {base_prompt['signature_name']}")
print(f"Predictor Type: {base_prompt['predictor_type']}")
print(f"Uses CoT: {base_prompt['uses_chain_of_thought']}")
print(f"Docstring Length: {len(base_prompt['docstring'])} chars")
print(f"Input Fields: {len(base_prompt['input_fields'])}")
print(f"Output Fields: {len(base_prompt['output_fields'])}")

if base_prompt['docstring']:
    print(f"\nSystem Prompt Preview:")
    print("-" * 80)
    print(base_prompt['docstring'][:800])
    if len(base_prompt['docstring']) > 800:
        print(f"\n... [truncated, total: {len(base_prompt['docstring'])} chars]")

if base_prompt['input_fields']:
    print(f"\nInput Fields:")
    for field, info in base_prompt['input_fields'].items():
        desc = info['description'][:100] if info['description'] else "No description"
        print(f"  • {field}: {desc}")

if base_prompt['output_fields']:
    print(f"\nOutput Fields:")
    for field, info in base_prompt['output_fields'].items():
        desc = info['description'][:100] if info['description'] else "No description"
        print(f"  • {field}: {desc}")

# =============================================================================
# Extract from Pattern-Enhanced Stitcher
# =============================================================================

print("\n" + "-"*80)
print("EXTRACTING: PATTERN-ENHANCED STITCHER")
print("-"*80)

enhanced_prompt = deep_inspect_predictor(pattern_enhanced_stitcher.generate_response, "Pattern-Enhanced Stitcher")

# Print summary
print(f"\n{'='*80}")
print("PATTERN-ENHANCED STITCHER SUMMARY")
print(f"{'='*80}")
print(f"Signature: {enhanced_prompt['signature_name']}")
print(f"Predictor Type: {enhanced_prompt['predictor_type']}")
print(f"Uses CoT: {enhanced_prompt['uses_chain_of_thought']}")
print(f"Docstring Length: {len(enhanced_prompt['docstring'])} chars")
print(f"Input Fields: {len(enhanced_prompt['input_fields'])}")
print(f"Output Fields: {len(enhanced_prompt['output_fields'])}")
print(f"Few-Shot Demos: {enhanced_prompt['num_demos']}")

if enhanced_prompt['docstring']:
    print(f"\nSystem Prompt Preview:")
    print("-" * 80)
    print(enhanced_prompt['docstring'][:800])
    if len(enhanced_prompt['docstring']) > 800:
        print(f"\n... [truncated, total: {len(enhanced_prompt['docstring'])} chars]")

if enhanced_prompt['input_fields']:
    print(f"\nInput Fields:")
    for field, info in enhanced_prompt['input_fields'].items():
        desc = info['description'][:100] if info['description'] else "No description"
        print(f"  • {field}: {desc}")

if enhanced_prompt['output_fields']:
    print(f"\nOutput Fields:")
    for field, info in enhanced_prompt['output_fields'].items():
        desc = info['description'][:100] if info['description'] else "No description"
        print(f"  • {field}: {desc}")

if enhanced_prompt['num_demos'] > 0:
    print(f"\nFew-Shot Demo Samples (showing first 3):")
    for demo in enhanced_prompt['few_shot_demos'][:3]:
        print(f"\n  Demo {demo['demo_id']}:")
        print(f"    Q: {demo['farmer_question'][:80]}...")
        print(f"    R: {demo['response'][:80]}...")
    if enhanced_prompt['num_demos'] > 3:
        print(f"\n  ... and {enhanced_prompt['num_demos'] - 3} more demos")

# =============================================================================
# Alternative: Get signature from the source code definitions
# =============================================================================

print("\n" + "="*80)
print("EXTRACTING SIGNATURES FROM SOURCE DEFINITIONS")
print("="*80)

try:
    # Try to get FactStitching signature directly
    print("\nLooking for FactStitching class...")
    if 'FactStitching' in globals():
        FactStitching = globals()['FactStitching']
        print(f"✓ Found FactStitching in globals")

        base_prompt["signature_name"] = "FactStitching"
        base_prompt["docstring"] = FactStitching.__doc__ or ""

        if hasattr(FactStitching, 'input_fields'):
            for name, field in FactStitching.input_fields.items():
                desc = ""
                try:
                    if hasattr(field, 'json_schema_extra') and field.json_schema_extra:
                        desc = field.json_schema_extra.get('desc', '')
                except:
                    pass
                base_prompt["input_fields"][name] = {"description": desc}

        if hasattr(FactStitching, 'output_fields'):
            for name, field in FactStitching.output_fields.items():
                desc = ""
                try:
                    if hasattr(field, 'json_schema_extra') and field.json_schema_extra:
                        desc = field.json_schema_extra.get('desc', '')
                except:
                    pass
                base_prompt["output_fields"][name] = {"description": desc}

        enhanced_prompt["signature_name"] = "FactStitching"
        enhanced_prompt["docstring"] = FactStitching.__doc__ or ""
        enhanced_prompt["input_fields"] = base_prompt["input_fields"].copy()
        enhanced_prompt["output_fields"] = base_prompt["output_fields"].copy()

        print(f"✓ Extracted FactStitching signature from source")
        print(f"  Docstring: {len(FactStitching.__doc__)} chars")
        print(f"  Input fields: {len(base_prompt['input_fields'])}")
        print(f"  Output fields: {len(base_prompt['output_fields'])}")

except Exception as e:
    print(f"❌ Error extracting from source: {e}")

# =============================================================================
# Update JSON
# =============================================================================

print("\n" + "="*80)
print("UPDATING JSON FILE")
print("="*80)

results_data["base_stitcher"] = {
    "prompt_configuration": base_prompt,
    **{k: v for k, v in results_data["base_stitcher"].items() if k != "prompt_configuration"}
}

results_data["pattern_enhanced_stitcher"] = {
    "prompt_configuration": enhanced_prompt,
    **{k: v for k, v in results_data["pattern_enhanced_stitcher"].items() if k != "prompt_configuration"}
}

results_data["prompt_extraction_timestamp"] = datetime.now().isoformat()

output_filename = "evaluation_results_20260112_085427_with_prompts.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(results_data, f, indent=2, ensure_ascii=False)

print(f"\n✓ JSON saved: {output_filename}")
print(f"  Base docstring: {len(base_prompt.get('docstring', ''))} chars")
print(f"  Enhanced docstring: {len(enhanced_prompt.get('docstring', ''))} chars")
print(f"  Enhanced demos: {enhanced_prompt.get('num_demos', 0)}")
print(f"  File size: {len(json.dumps(results_data)) / 1024:.1f} KB")

# =============================================================================
# Save to text files
# =============================================================================

print("\n" + "-"*80)
print("SAVING TEXT FILES")
print("-"*80)

with open("base_stitcher_prompt.txt", "w", encoding="utf-8") as f:
    f.write("="*80 + "\n")
    f.write("BASE STITCHER CONFIGURATION\n")
    f.write("="*80 + "\n\n")
    f.write(f"Signature: {base_prompt.get('signature_name', 'N/A')}\n")
    f.write(f"Predictor: {base_prompt.get('predictor_type', 'N/A')}\n")
    f.write(f"Chain-of-Thought: {base_prompt.get('uses_chain_of_thought', False)}\n\n")
    f.write("SYSTEM PROMPT:\n" + "-"*80 + "\n")
    f.write(base_prompt.get('docstring', 'Not found'))
    f.write("\n\n" + "-"*80 + "\n\nINPUT FIELDS:\n")
    for field, info in base_prompt.get('input_fields', {}).items():
        f.write(f"\n{field}:\n  {info.get('description', 'N/A')}\n")
    f.write("\n\nOUTPUT FIELDS:\n")
    for field, info in base_prompt.get('output_fields', {}).items():
        f.write(f"\n{field}:\n  {info.get('description', 'N/A')}\n")

print("✓ base_stitcher_prompt.txt")

with open("pattern_enhanced_stitcher_prompt.txt", "w", encoding="utf-8") as f:
    f.write("="*80 + "\n")
    f.write("PATTERN-ENHANCED STITCHER CONFIGURATION\n")
    f.write("="*80 + "\n\n")
    f.write(f"Signature: {enhanced_prompt.get('signature_name', 'N/A')}\n")
    f.write(f"Predictor: {enhanced_prompt.get('predictor_type', 'N/A')}\n")
    f.write(f"Chain-of-Thought: {enhanced_prompt.get('uses_chain_of_thought', False)}\n")
    f.write(f"Few-Shot Demos: {enhanced_prompt.get('num_demos', 0)}\n\n")
    f.write("SYSTEM PROMPT:\n" + "-"*80 + "\n")
    f.write(enhanced_prompt.get('docstring', 'Not found'))
    f.write("\n\n" + "-"*80 + "\n\nINPUT FIELDS:\n")
    for field, info in enhanced_prompt.get('input_fields', {}).items():
        f.write(f"\n{field}:\n  {info.get('description', 'N/A')}\n")
    f.write("\n\nOUTPUT FIELDS:\n")
    for field, info in enhanced_prompt.get('output_fields', {}).items():
        f.write(f"\n{field}:\n  {info.get('description', 'N/A')}\n")
    f.write("\n\n" + "="*80 + "\nFEW-SHOT DEMOS\n" + "="*80 + "\n")
    for demo in enhanced_prompt.get('few_shot_demos', []):
        f.write(f"\n{'='*80}\nDEMO {demo['demo_id']}\n{'='*80}\n\n")
        f.write(f"Q: {demo['farmer_question']}\n\n")
        f.write(f"FACTS: {demo['agricultural_facts']}\n\n")
        f.write(f"CONTEXT: {demo['crop_context']}\n\n")
        f.write(f"R: {demo['response']}\n\n")

print("✓ pattern_enhanced_stitcher_prompt.txt")

print("\n" + "="*80)
print("DONE!")
print("="*80)


EXTRACTING PROMPTS FROM MODELS

--------------------------------------------------------------------------------
EXTRACTING: BASE STITCHER
--------------------------------------------------------------------------------

DEEP INSPECTION: Base Stitcher

1. Checking .predictors()...
   Found 1 sub-predictors
   Inner predictor type: Predict
   ✓ Found signature in inner predictor!

2. Checking for few-shot demos...
   No demos found

3. Checking prediction history...
   Found 1 history entries
   Last entry type: <class 'dict'>

BASE STITCHER SUMMARY
Signature: FactStitching
Predictor Type: Predict
Uses CoT: False
Docstring Length: 2260 chars
Input Fields: 3
Output Fields: 1

System Prompt Preview:
--------------------------------------------------------------------------------
You are Farmer.CHAT, a knowledgeable agricultural advisor helping farmers.

**YOUR ROLE:**
- Experienced agricultural extension worker with deep local knowledge
- Supportive mentor who understands smallholder far

In [ ]:
# =============================================================================
# Add Final Results Summary at the Top of JSON
# =============================================================================

import json
from datetime import datetime

# Load the existing JSON
with open('evaluation_results_20260112_085427_with_prompts.json', 'r', encoding='utf-8') as f:
    results_data = json.load(f)

print("\n" + "="*80)
print("ADDING FINAL RESULTS SUMMARY AT TOP OF JSON")
print("="*80)

# =============================================================================
# Calculate comprehensive statistics
# =============================================================================

# Basic scores
base_scores = results_data["base_stitcher"]["scores"]
enhanced_scores = results_data["pattern_enhanced_stitcher"]["scores"]
num_examples = results_data["num_examples"]

base_avg = results_data["base_stitcher"]["average_score"]
enhanced_avg = results_data["pattern_enhanced_stitcher"]["average_score"]

# Improvement metrics
improvement = enhanced_avg - base_avg
improvement_pct = (improvement / base_avg) * 100 if base_avg > 0 else 0

# Head-to-head comparison
wins = sum(1 for i in range(num_examples) if enhanced_scores[i] > base_scores[i])
losses = sum(1 for i in range(num_examples) if enhanced_scores[i] < base_scores[i])
ties = sum(1 for i in range(num_examples) if enhanced_scores[i] == base_scores[i])

# Dimension-wise breakdown
dimensions = ['content_quality', 'communication_style', 'practical_advice',
              'safety_credibility', 'conversation_flow', 'response_format']

dimension_stats = {}
for dim in dimensions:
    base_dim_scores = []
    enhanced_dim_scores = []

    for ex in results_data["base_stitcher"]["examples"]:
        score = ex['evaluation'].get(dim, {}).get('score', 3)
        base_dim_scores.append(score)

    for ex in results_data["pattern_enhanced_stitcher"]["examples"]:
        score = ex['evaluation'].get(dim, {}).get('score', 3)
        enhanced_dim_scores.append(score)

    base_dim_avg = sum(base_dim_scores) / len(base_dim_scores) if base_dim_scores else 0
    enhanced_dim_avg = sum(enhanced_dim_scores) / len(enhanced_dim_scores) if enhanced_dim_scores else 0
    dim_improvement = enhanced_dim_avg - base_dim_avg

    dimension_stats[dim] = {
        "base_average": round(base_dim_avg, 2),
        "enhanced_average": round(enhanced_dim_avg, 2),
        "improvement": round(dim_improvement, 2),
        "improvement_percentage": round((dim_improvement / base_dim_avg * 100) if base_dim_avg > 0 else 0, 2)
    }

# Distribution analysis
def get_score_distribution(scores):
    distribution = {
        "5.0": 0,
        "4.5-4.9": 0,
        "4.0-4.4": 0,
        "3.5-3.9": 0,
        "3.0-3.4": 0,
        "below_3.0": 0
    }
    for score in scores:
        if score >= 5.0:
            distribution["5.0"] += 1
        elif score >= 4.5:
            distribution["4.5-4.9"] += 1
        elif score >= 4.0:
            distribution["4.0-4.4"] += 1
        elif score >= 3.5:
            distribution["3.5-3.9"] += 1
        elif score >= 3.0:
            distribution["3.0-3.4"] += 1
        else:
            distribution["below_3.0"] += 1
    return distribution

base_distribution = get_score_distribution(base_scores)
enhanced_distribution = get_score_distribution(enhanced_scores)

# Top improvements and regressions
score_differences = []
for i in range(num_examples):
    diff = enhanced_scores[i] - base_scores[i]
    score_differences.append({
        "example_id": i,
        "question": results_data["pattern_enhanced_stitcher"]["examples"][i]["question"][:100] + "...",
        "base_score": round(base_scores[i], 2),
        "enhanced_score": round(enhanced_scores[i], 2),
        "difference": round(diff, 2)
    })

# Sort by difference
score_differences.sort(key=lambda x: x["difference"], reverse=True)
top_improvements = score_differences[:5]
top_regressions = score_differences[-5:]

# Statistical measures
import math

def calculate_std_dev(scores):
    if not scores:
        return 0
    mean = sum(scores) / len(scores)
    variance = sum((x - mean) ** 2 for x in scores) / len(scores)
    return math.sqrt(variance)

base_std = calculate_std_dev(base_scores)
enhanced_std = calculate_std_dev(enhanced_scores)

# =============================================================================
# Build final results summary
# =============================================================================

final_results = {
    "evaluation_metadata": {
        "timestamp": results_data.get("timestamp", "N/A"),
        "prompt_extraction_timestamp": results_data.get("prompt_extraction_timestamp", "N/A"),
        "num_examples_tested": num_examples,
        "models_compared": [
            results_data["base_stitcher"]["prompt_configuration"].get("signature_name", "Base Stitcher"),
            results_data["pattern_enhanced_stitcher"]["prompt_configuration"].get("signature_name", "Pattern-Enhanced Stitcher")
        ]
    },

    "overall_performance": {
        "base_stitcher": {
            "average_score": round(base_avg, 2),
            "min_score": round(min(base_scores), 2),
            "max_score": round(max(base_scores), 2),
            "std_deviation": round(base_std, 2)
        },
        "pattern_enhanced_stitcher": {
            "average_score": round(enhanced_avg, 2),
            "min_score": round(min(enhanced_scores), 2),
            "max_score": round(max(enhanced_scores), 2),
            "std_deviation": round(enhanced_std, 2)
        },
        "improvement": {
            "absolute": round(improvement, 2),
            "percentage": round(improvement_pct, 2),
            "is_better": improvement > 0
        }
    },

    "head_to_head_comparison": {
        "pattern_enhanced_wins": wins,
        "pattern_enhanced_losses": losses,
        "ties": ties,
        "win_rate_percentage": round((wins / num_examples * 100), 2),
        "summary": f"Pattern-Enhanced won {wins}/{num_examples} examples ({round(wins/num_examples*100, 1)}%)"
    },

    "dimension_breakdown": dimension_stats,

    "score_distribution": {
        "base_stitcher": base_distribution,
        "pattern_enhanced_stitcher": enhanced_distribution
    },

    "top_improvements": top_improvements,

    "top_regressions": top_regressions,

    "key_findings": {
        "best_performing_dimension": max(dimension_stats.items(), key=lambda x: x[1]["improvement"])[0],
        "worst_performing_dimension": min(dimension_stats.items(), key=lambda x: x[1]["improvement"])[0],
        "consistency": {
            "base_consistency": "Low" if base_std > 0.5 else "Medium" if base_std > 0.3 else "High",
            "enhanced_consistency": "Low" if enhanced_std > 0.5 else "Medium" if enhanced_std > 0.3 else "High",
            "note": "Lower std deviation = more consistent scores"
        },
        "overall_verdict": "Pattern-Enhanced performs better" if improvement > 0 else "Base performs better" if improvement < 0 else "Performance is tied"
    }
}

# =============================================================================
# Restructure JSON with final_results at the top
# =============================================================================

# Create new ordered dictionary with final_results first
restructured_data = {
    "final_results": final_results
}

# Add all other existing data
for key, value in results_data.items():
    if key != "final_results":  # Avoid duplication if it already exists
        restructured_data[key] = value

# =============================================================================
# Save updated JSON
# =============================================================================

output_filename = "evaluation_results_20260108_130441_FINAL.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(restructured_data, f, indent=2, ensure_ascii=False)

print(f"\n✓ Updated JSON saved: {output_filename}")
print(f"  File size: {len(json.dumps(restructured_data)) / 1024:.1f} KB")

# =============================================================================
# Print summary to console
# =============================================================================

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)

print(f"\n📊 OVERALL PERFORMANCE:")
print(f"  Base Stitcher:           {base_avg:.2f}/5.0 (σ={base_std:.2f})")
print(f"  Pattern-Enhanced:        {enhanced_avg:.2f}/5.0 (σ={enhanced_std:.2f})")
print(f"  Improvement:             {improvement:+.2f} ({improvement_pct:+.1f}%)")

print(f"\n🏆 HEAD-TO-HEAD:")
print(f"  Pattern-Enhanced Wins:   {wins}/{num_examples} ({wins/num_examples*100:.1f}%)")
print(f"  Pattern-Enhanced Losses: {losses}/{num_examples} ({losses/num_examples*100:.1f}%)")
print(f"  Ties:                    {ties}/{num_examples} ({ties/num_examples*100:.1f}%)")

print(f"\n📈 DIMENSION BREAKDOWN:")
for dim, stats in dimension_stats.items():
    dim_name = dim.replace('_', ' ').title()
    print(f"  {dim_name:25s} Base: {stats['base_average']:.2f} → Enhanced: {stats['enhanced_average']:.2f} (Δ {stats['improvement']:+.2f})")

print(f"\n🎯 KEY FINDINGS:")
print(f"  Best Dimension:          {final_results['key_findings']['best_performing_dimension'].replace('_', ' ').title()}")
print(f"  Worst Dimension:         {final_results['key_findings']['worst_performing_dimension'].replace('_', ' ').title()}")
print(f"  Base Consistency:        {final_results['key_findings']['consistency']['base_consistency']}")
print(f"  Enhanced Consistency:    {final_results['key_findings']['consistency']['enhanced_consistency']}")
print(f"  Overall Verdict:         {final_results['key_findings']['overall_verdict']}")

print(f"\n⬆️  TOP 3 IMPROVEMENTS:")
for i, item in enumerate(top_improvements[:3], 1):
    print(f"  {i}. Example {item['example_id']}: {item['base_score']:.2f} → {item['enhanced_score']:.2f} (Δ {item['difference']:+.2f})")
    print(f"     Q: {item['question']}")

if top_regressions and any(r['difference'] < 0 for r in top_regressions):
    print(f"\n⬇️  TOP 3 REGRESSIONS:")
    for i, item in enumerate([r for r in top_regressions if r['difference'] < 0][:3], 1):
        print(f"  {i}. Example {item['example_id']}: {item['base_score']:.2f} → {item['enhanced_score']:.2f} (Δ {item['difference']:+.2f})")
        print(f"     Q: {item['question']}")

print("\n" + "="*80)
print("✅ COMPLETE!")
print("="*80)
print(f"\nFinal results now at the TOP of: {output_filename}")


ADDING FINAL RESULTS SUMMARY AT TOP OF JSON


IndexError: list index out of range